In [1]:
import sys
sys.path.append("..")  # needed to import dataloader.py
import torch
import torchvision
from dataloader import MultiTaskBrainDataset
from torch.utils.data import DataLoader
from torchinfo import summary
torch.manual_seed(42)  # set seed for reproducibility
device = torch.device('cpu')

In [2]:
class BasicEncoderDecoder(torch.nn.Module):
    """
    Minimal encoder-decoder for brain tumor segmentation.

    Architecture:
        Encoder: 3 conv blocks, each halving spatial dims via stride-2 conv.
        Bottleneck: 1x1 conv to compress the feature map.
        Decoder: 3 transpose-conv blocks, each doubling spatial dims back up.
        Head: 1x1 conv to map to num_classes per pixel.
    """

    def __init__(self, num_classes=1):
        super(BasicEncoderDecoder, self).__init__()

        self.enc1 = torch.nn.Sequential(
            torch.nn.Conv2d(1,  16, kernel_size=3, stride=2, padding=1),  
            torch.nn.ReLU(),
            torch.nn.Dropout(0.25),
        )
        self.enc2 = torch.nn.Sequential(
            torch.nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),  
            torch.nn.ReLU(),
            torch.nn.Dropout(0.25),
        )
        self.enc3 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  
            torch.nn.ReLU(),
            torch.nn.Dropout(0.25),
        )


        self.bottleneck = torch.nn.Sequential(
            torch.nn.Conv2d(64, 64, kernel_size=1),
            torch.nn.ReLU(),
        )


        self.dec1 = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),    
            torch.nn.ReLU(),
            torch.nn.Dropout(0.25),
        )
        self.dec2 = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),    
            torch.nn.ReLU(),
            torch.nn.Dropout(0.25),
        )
        self.dec3 = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(16, 16, kernel_size=2, stride=2),    
            torch.nn.ReLU(),
            torch.nn.Dropout(0.25),
        )

        self.head = torch.nn.Conv2d(16, num_classes, kernel_size=1)

    def forward(self, x):
        # Encode
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decode
        x = self.dec1(x)
        x = self.dec2(x)
        x = self.dec3(x)

        # Per-pixel logits
        x = self.head(x)
        return x
    
class DiceBCELoss(torch.nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, eps=1e-6, pos_weight=10.0):
        super().__init__()
        self.bce_weight  = bce_weight
        self.dice_weight = dice_weight
        self.eps         = eps
        self.pos_weight  = pos_weight

    def forward(self, pred_logits, target):
        probs = torch.sigmoid(pred_logits)

        # Weighted BCE
        bce = -(target * torch.log(probs + 1e-6) * self.pos_weight +
                (1 - target) * torch.log(1 - probs + 1e-6))
        bce_loss = bce.mean()

        # Dice
        intersection = (probs * target).sum(dim=(1,2,3))
        union        = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
        dice_loss    = 1 - (2*intersection + self.eps) / (union + self.eps)
        dice_loss    = dice_loss.mean()

        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

In [3]:
model = BasicEncoderDecoder(num_classes=1)  # binary: tumor vs background
print(summary(model, input_size=(16, 1, 224, 224)))
model.to(device)

Layer (type:depth-idx)                   Output Shape              Param #
BasicEncoderDecoder                      [16, 1, 224, 224]         --
├─Sequential: 1-1                        [16, 16, 112, 112]        --
│    └─Conv2d: 2-1                       [16, 16, 112, 112]        160
│    └─ReLU: 2-2                         [16, 16, 112, 112]        --
│    └─Dropout: 2-3                      [16, 16, 112, 112]        --
├─Sequential: 1-2                        [16, 32, 56, 56]          --
│    └─Conv2d: 2-4                       [16, 32, 56, 56]          4,640
│    └─ReLU: 2-5                         [16, 32, 56, 56]          --
│    └─Dropout: 2-6                      [16, 32, 56, 56]          --
├─Sequential: 1-3                        [16, 64, 28, 28]          --
│    └─Conv2d: 2-7                       [16, 64, 28, 28]          18,496
│    └─ReLU: 2-8                         [16, 64, 28, 28]          --
│    └─Dropout: 2-9                      [16, 64, 28, 28]          --
├─Seque

BasicEncoderDecoder(
  (enc1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
  )
  (enc2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
  )
  (enc3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
  )
  (bottleneck): Sequential(
    (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU()
  )
  (dec1): Sequential(
    (0): ConvTranspose2d(64, 32, kernel_size=(2, 2), stride=(2, 2))
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
  )
  (dec2): Sequential(
    (0): ConvTranspose2d(32, 16, kernel_size=(2, 2), stride=(2, 2))
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
  )
  (dec3): Sequential(
    (0): ConvTranspose2d(16, 16, kernel_size=(2, 2), stride=(2, 2))
    (1): ReL

In [ ]:
TRAIN = True # Toggle this to False if you only want the test set results without training using a saved model

if TRAIN:    
    from torch.utils.data import DataLoader
    import os
    import json

    train_data = MultiTaskBrainDataset(
        os.path.join("..", "brisc2025"),
        mode="segmentation",
        train_or_test="train",
        subset="train",
        augment=True,
        seed=42,
        new_height=256, 
        new_width=256
    )

    val_data = MultiTaskBrainDataset(
        os.path.join("..", "brisc2025"),
        mode="segmentation",
        train_or_test="train",
        subset="val",
        augment=False,
        seed=42,
        new_height=256, 
        new_width=256
    )

    train_loader = DataLoader(train_data, batch_size=32, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_data,   batch_size=32, shuffle=False, num_workers=0)

    # BCEWithLogitsLoss for binary segmentation (num_classes=1).
    # Switch to CrossEntropyLoss if you bump num_classes > 1.
    loss_fn   = DiceBCELoss(bce_weight=0.5, dice_weight=0.5)
    optimizer = torch.optim.Adam(model.parameters())

    def dice_coefficient(pred_logits, target, threshold=0.5, eps=1e-6):
        """
        Computes the Dice score between predicted binary mask and ground truth.
        pred_logits: raw model output (B, 1, H, W)
        target:      ground truth mask (B, 1, H, W), values in {0, 1}
        """
        probs = torch.sigmoid(pred_logits)
        preds = (probs > threshold).float()
        intersection = (preds * target).sum(dim=(1, 2, 3))
        union = preds.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        dice = (2.0 * intersection + eps) / (union + eps)
        return dice.mean().item()

    best_val_dice  = 0.0
    checkpoint_dir = os.path.join("..", "results", "segmentation", "tumor_segmentation_v1_grant")
    os.makedirs(checkpoint_dir, exist_ok=True)

    best_model_path = os.path.join(checkpoint_dir, "best_model.pt")
    metadata_path   = os.path.join(checkpoint_dir, "training_metadata.json")

    training_metadata = {
        "best_val_dice": 0.0,
        "best_epoch":    None,
        "history":       []
    }

    for epoch in range(100):
        model.train()
        running_loss = 0.0
        running_dice = 0.0

        for inputs, masks in train_loader:
            inputs, masks = inputs.to(device), masks.float().to(device)

            optimizer.zero_grad()
            predictions = model(inputs)
            loss = loss_fn(predictions, masks)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            running_dice += dice_coefficient(predictions, masks)

        epoch_loss = running_loss / len(train_loader)
        epoch_dice = running_dice / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0
        val_dice = 0.0

        with torch.no_grad():
            for inputs, masks in val_loader:
                inputs, masks = inputs.to(device), masks.float().to(device)
                predictions = model(inputs)
                loss = loss_fn(predictions, masks.float())
                val_loss += loss.item()
                val_dice += dice_coefficient(predictions, masks)

        val_loss /= len(val_loader)
        val_dice /= len(val_loader)

        print(
            f"Epoch {epoch+1:03d} | "
            f"Train Loss: {epoch_loss:.4f} | Train Dice: {epoch_dice:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}"
        )

        # Record epoch metrics in metadata
        training_metadata["history"].append({
            "epoch":      epoch + 1,
            "train_loss": round(epoch_loss, 6),
            "train_dice": round(epoch_dice, 6),
            "val_loss":   round(val_loss,   6),
            "val_dice":   round(val_dice,   6),
        })

        # Save best model and update metadata if improved
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            training_metadata["best_val_dice"] = round(val_dice, 6)
            training_metadata["best_epoch"]    = epoch + 1

            torch.save({
                "epoch":                epoch + 1,
                "model_state_dict":     model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice":             val_dice,
                "val_loss":             val_loss,
            }, best_model_path)
            print(f"  -> Best model saved (epoch {epoch+1}, val_dice={val_dice:.4f})")

        # Write metadata JSON after every epoch
        with open(metadata_path, "w") as f:
            json.dump(training_metadata, f, indent=4)

In [8]:
from dataloader import MultiTaskBrainDataset
import os
import numpy as np

test_data = MultiTaskBrainDataset(
    os.path.join("..", "brisc2025"),
    mode="segmentation",
    train_or_test="test",
    augment=False,
    seed=42,
    new_height=256,
    new_width=256,
)

# Load v1 checkpoint
checkpoint_dir  = os.path.join("..", "results", "segmentation", "tumor_segmentation_v1_grant")
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")

loss_fn   = DiceBCELoss(bce_weight=0.5, dice_weight=0.5)

checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()
print(f"Loaded v1 checkpoint from epoch {checkpoint['epoch']} (val_dice={checkpoint['val_dice']:.4f})")

# Build per-sample metadata (tumor type inferred from path)
PREFIX_MAP = {"gl": "glioma", "me": "meningioma", "pi": "pituitary"}

def infer_tumor_type(img_path, mask_path):
    if mask_path is None:
        return "no_tumor"
    parts = os.path.splitext(os.path.basename(img_path))[0].lower().split("_")
    for p in parts:
        if p in PREFIX_MAP:
            return PREFIX_MAP[p]
    return "tumor"

sample_meta = [
    (img_path, mask_path, infer_tumor_type(img_path, mask_path))
    for img_path, mask_path in test_data.samples
]

# Run inference sample-by-sample to track per-class Dice
loader = DataLoader(test_data, batch_size=1, shuffle=False, num_workers=0)

def dice_coefficient(pred_logits, target, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(pred_logits)
    preds = (probs > threshold).float()
    intersection = (preds * target).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return ((2.0 * intersection + eps) / (union + eps)).mean().item()

from collections import defaultdict
from sklearn.metrics import precision_score, recall_score, f1_score

test_loss = 0.0
test_dice = 0.0
all_preds = []
all_masks = []
class_dice = defaultdict(list)

with torch.no_grad():
    for idx, (inputs, masks) in enumerate(loader):
        inputs, masks = inputs.to(device), masks.to(device)
        predictions = model(inputs)

        loss = loss_fn(predictions, masks.float())
        test_loss += loss.item()
        test_dice += dice_coefficient(predictions, masks)

        binary_preds = (torch.sigmoid(predictions) > 0.5).float().cpu()
        all_preds.append(binary_preds)
        all_masks.append(masks.cpu())

        tumor_type = sample_meta[idx][2]
        class_dice[tumor_type].append(dice_coefficient(predictions, masks))

test_loss /= len(loader)
test_dice /= len(loader)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test Dice : {test_dice:.4f}\n")

# Per-class Dice
print(f"{'Tumor Type':<22} {'N':>5} {'Mean Dice':>10} {'Std Dice':>10} {'Min':>8} {'Max':>8}")
print("-" * 68)
for tumor_type in ["glioma", "meningioma", "pituitary", "no_tumor"]:
    dices = class_dice.get(tumor_type, [])
    if not dices:
        continue
    d = np.array(dices)
    print(f"{tumor_type:<22} {len(d):>5} {d.mean():>10.4f} {d.std():>10.4f} {d.min():>8.4f} {d.max():>8.4f}")
print("-" * 68)
all_dices = [d for dices in class_dice.values() for d in dices]
d = np.array(all_dices)
print(f"{'OVERALL':<22} {len(d):>5} {d.mean():>10.4f} {d.std():>10.4f} {d.min():>8.4f} {d.max():>8.4f}")

# Pixel-level metrics
all_preds_flat = torch.cat(all_preds).view(-1).numpy().astype(int)
all_masks_flat = torch.cat(all_masks).view(-1).numpy().astype(int)

precision = precision_score(all_masks_flat, all_preds_flat, zero_division=0)
recall    = recall_score   (all_masks_flat, all_preds_flat, zero_division=0)
f1        = f1_score       (all_masks_flat, all_preds_flat, zero_division=0)

print(f"\nPixel Precision : {precision:.4f}")
print(f"Pixel Recall    : {recall:.4f}")
print(f"Pixel F1 Score  : {f1:.4f}")

C:\Users\glawl\AppData\Local\Temp\ipykernel_21988\3021359052.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(best_model_path, map_location=devic

Loaded v1 checkpoint from epoch 92 (val_dice=0.3268)
Test Loss : 0.5094
Test Dice : 0.3554

Tumor Type                 N  Mean Dice   Std Dice      Min      Max
--------------------------------------------------------------------
glioma                   254     0.2375     0.2087   0.0000   0.8715
meningioma               306     0.5826     0.2210   0.0000   0.9709
pituitary                300     0.3859     0.2294   0.0000   0.8716
no_tumor                 140     0.0071     0.0842   0.0000   1.0000
--------------------------------------------------------------------
OVERALL                 1000     0.3554     0.2817   0.0000   1.0000

Pixel Precision : 0.3255
Pixel Recall    : 0.5037
Pixel F1 Score  : 0.3955
